# ONDO 가상 피팅 이미지 일괄 생성 (CatVTON · Kaggle 무료 GPU)

**한 장에 한 벌씩만** 입힙니다. 상의·하의·아우터·신발·가방을 조합하지 않습니다.

모든 결과는 **기본 모델 6장(`{체형}_{톤}_lower_base.png`)** 위에 생성합니다. 모델이 직접 입고·신고·드는 이미지로 만듭니다.

| 카테고리 | 마스크 |
|---|---|
| 상의 | upper (크롭탑 자리) |
| 아우터 | outer |
| 하의(스커트·팬츠·쇼츠) | lower → 쇼츠까지 덮어서 치마는 맨다리 위에 |
| 신발 | 발 영역 (맨발 위에 신긴 모습으로 생성) |
| 가방 | 허리 옆·손 영역 (모델이 든 모습으로 생성) |

## 실행 방법
1. 오른쪽 **Settings → Accelerator: GPU T4 x2(또는 P100)**, **Internet: On** (전화번호 인증 필요)
2. **Add Input**으로 `ondo-tryon-kit` 데이터셋(이 zip을 올린 것)을 추가
3. 아래 `설정` 셀에서 만들 범위를 고르고 **Run All**
4. 끝나면 오른쪽 **Output**에서 `ondo_fitting_results.zip`을 내려받아 Claude에게 전달

세션이 끊겨도 이어서 만들 수 있습니다. 이전 실행의 Output을 Input으로 추가하면 이미 만든 조합은 건너뜁니다.

In [ ]:
# ===== 설정 =====
PRIORITY_MAX = 1            # 1 = 상품마다 대표 색상 1개만 (추천, 약 990장). 2 = 톤에 맞는 모든 색상
CATEGORIES = ["top", "bottom", "outer", "shoes", "bag"]
BODIES = ["straight", "wave", "natural"]
TONES = ["warm", "cool"]
STEPS = 30                  # 품질↑ 50, 속도↑ 25
GUIDANCE = 2.5
SEED = 42
LIMIT = None                # 테스트할 때는 6 정도로
WIDTH, HEIGHT = 768, 1024
TEST_IDS = None             # 시험용 job_id 목록 (None이면 전체)


In [ ]:
import subprocess, sys, os
if not os.path.exists("/kaggle/working/CatVTON"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zheng-Chong/CatVTON.git", "/kaggle/working/CatVTON"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.30.0", "accelerate>=0.31.0", "fvcore", "cloudpickle", "omegaconf", "pycocotools", "av", "peft", "rembg", "onnxruntime"], check=True)
sys.path.insert(0, "/kaggle/working/CatVTON")
print("ready")

In [ ]:
WORKER_SRC = "import os, sys, csv, glob, time, json\nSHARD, NSHARD = int(sys.argv[1]), int(sys.argv[2])\ncfg = json.load(open(\"/kaggle/working/run_config.json\"))\nglobals().update(cfg)\nsys.path.insert(0, \"/kaggle/working/CatVTON\")\nimport glob, csv\nkit = os.path.dirname(glob.glob(\"/kaggle/input/**/jobs.csv\", recursive=True)[0])\nOUT = \"/kaggle/working/results\"; os.makedirs(OUT, exist_ok=True)\ndone = set()\nfor p in glob.glob(\"/kaggle/input/**/results/*/*.png\", recursive=True) + glob.glob(OUT + \"/*/*.png\"):\n    done.add(os.path.basename(os.path.dirname(p)) + \"_\" + os.path.splitext(os.path.basename(p))[0])\njobs = [r for r in csv.DictReader(open(f\"{kit}/jobs.csv\", encoding=\"utf-8\"))\n        if int(r[\"priority\"]) <= PRIORITY_MAX and r[\"category\"] in CATEGORIES and r[\"body\"] in BODIES and r[\"tone\"] in TONES]\ntodo = [r for r in jobs if r[\"job_id\"] not in done]\nif TEST_IDS: todo = [r for r in todo if r[\"job_id\"] in TEST_IDS]\nif LIMIT: todo = todo[:LIMIT]\ntodo = todo[SHARD::NSHARD]\nprint(f\"[gpu{SHARD}] kit={kit}  대상 {len(jobs)}장 / 이미 완료 {len(jobs)-len(todo)}장 / 이번에 생성 {len(todo)}장\")\nimport torch, numpy as np\nfrom PIL import Image, ImageFilter\nfrom huggingface_hub import snapshot_download\nfrom diffusers.image_processor import VaeImageProcessor\nfrom model.pipeline import CatVTONPipeline\nfrom model.cloth_masker import AutoMasker\nfrom utils import resize_and_padding\n\nrepo = snapshot_download(\"zhengchong/CatVTON\")\npipe = CatVTONPipeline(base_ckpt=\"booksforcharlie/stable-diffusion-inpainting\", attn_ckpt=repo, attn_ckpt_version=\"mix\",\n                       weight_dtype=torch.float16, use_tf32=False, device=\"cuda\")\nmasker = AutoMasker(densepose_ckpt=os.path.join(repo, \"DensePose\"), schp_ckpt=os.path.join(repo, \"SCHP\"), device=\"cuda\")\nmask_proc = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)\nfrom rembg import remove, new_session\ncutter = new_session(\"u2net_human_seg\")\nprint(\"models loaded\")\nBG = (255, 255, 255)\ndef on_white(im):\n    im = im.convert(\"RGBA\"); bg = Image.new(\"RGBA\", im.size, BG + (255,)); bg.alpha_composite(im); return bg.convert(\"RGB\")\n\n_avatar_cache = {}\ndef avatar(name):\n    if name not in _avatar_cache:\n        rgba = Image.open(f\"{kit}/avatars/{name}\").convert(\"RGBA\")\n        _avatar_cache[name] = (rgba, on_white(rgba).resize((WIDTH, HEIGHT), Image.LANCZOS))\n    return _avatar_cache[name]\n\ndef body_alpha(rgba):\n    return (np.array(rgba.resize((WIDTH, HEIGHT)).split()[-1]) > 40)\n\ndef rect_mask(box):\n    m = np.zeros((HEIGHT, WIDTH), np.uint8); x0, y0, x1, y1 = [int(v) for v in box]\n    m[max(0,y0):min(HEIGHT,y1), max(0,x0):min(WIDTH,x1)] = 255; return Image.fromarray(m)\n\ndef accessory_mask(rgba, kind):\n    a = body_alpha(rgba); ys, xs = np.where(a); top, bot = ys.min(), ys.max(); H = bot - top\n    if kind == \"shoes\":                       # 발 영역 (발목 아래)\n        band = a[bot - int(H*.07):bot + 1]; cols = np.where(band.any(0))[0]\n        return rect_mask((cols.min() - 18, bot - int(H*.075), cols.max() + 18, min(HEIGHT, bot + 14)))\n    yh = top + int(H*.50); row = np.where(a[yh])[0]   # 가방: 허리 옆, 손 주변\n    return rect_mask((row.min() - int(H*.13), yh - int(H*.04), row.min() + int(H*.05), yh + int(H*.16)))\n\ndef make_mask(job, rgba, person):\n    gt = job[\"garment_type\"]\n    if gt in (\"upper\", \"outer\", \"lower\"):\n        m = masker(person, gt)[\"mask\"]\n        if gt == \"lower\":\n            # 맨다리 기본 모델의 쇼츠 영역이 남지 않도록 쇼츠+다리 전체를 확실히 덮는다 (치마는 맨다리 위에)\n            arr = np.array(m.convert(\"L\")) > 127; a = body_alpha(rgba); ys, _ = np.where(a); top, bot = ys.min(), ys.max(); H = bot - top\n            hip = top + int(H*.43); knee_floor = top + int(H*(.62 if job[\"is_skirt\"] == \"1\" else .9))\n            arr[hip:knee_floor] |= a[hip:knee_floor]\n            m = Image.fromarray((arr * 255).astype(np.uint8))\n        return m\n    return accessory_mask(rgba, gt)\nimport time, zipfile\nmanifest_path = f\"{OUT}/manifest_{SHARD}.csv\"\nfields = [\"job_id\",\"body\",\"tone\",\"variant_id\",\"product_id\",\"category\",\"subtype\",\"name\",\"color\",\"file\"]\nnew_file = not os.path.exists(manifest_path)\nmf = open(manifest_path, \"a\", newline=\"\", encoding=\"utf-8\"); mw = csv.DictWriter(mf, fieldnames=fields)\nif new_file: mw.writeheader()\ngen = torch.Generator(device=\"cuda\").manual_seed(SEED)\nt0 = time.time()\nfor i, job in enumerate(todo, 1):\n    try:\n        rgba, person = avatar(job[\"base_avatar\"])\n        cloth = resize_and_padding(on_white(Image.open(f\"{kit}/{job['image_path']}\")), (WIDTH, HEIGHT))\n        mask = mask_proc.blur(make_mask(job, rgba, person), blur_factor=9)\n        result = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=gen)[0]\n        result = result.resize(rgba.size, Image.LANCZOS)\n        cut = remove(result, session=cutter)              # 배경 제거 → 사이트 배경 위에 올리기\n        folder = f\"{OUT}/{job['body']}_{job['tone']}\"; os.makedirs(folder, exist_ok=True)\n        path = f\"{folder}/{job['variant_id']}.png\"; cut.save(path, optimize=True)\n        mw.writerow({k: job[k] for k in fields if k != \"file\"} | {\"file\": os.path.relpath(path, OUT)}); mf.flush()\n    except Exception as e:\n        print(\"실패\", job[\"job_id\"], e)\n    if i % 10 == 0 or i == len(todo):\n        el = time.time() - t0; print(f\"[gpu{SHARD}] {i}/{len(todo)}  {el/i:.1f}s/장  남은 예상 {el/i*(len(todo)-i)/60:.0f}분\")\nmf.close()\n"

In [ ]:
# ===== GPU 여러 장에 나눠서 동시에 생성 =====
import json, subprocess, torch
json.dump({k: v for k, v in dict(PRIORITY_MAX=PRIORITY_MAX, CATEGORIES=CATEGORIES, BODIES=BODIES, TONES=TONES, STEPS=STEPS, GUIDANCE=GUIDANCE, SEED=SEED, LIMIT=LIMIT, TEST_IDS=TEST_IDS, WIDTH=WIDTH, HEIGHT=HEIGHT).items()}, open("/kaggle/working/run_config.json", "w"))
open("/kaggle/working/worker.py", "w", encoding="utf-8").write(WORKER_SRC)
n = max(1, torch.cuda.device_count()); print("GPU", n, "장")
procs = [subprocess.Popen([sys.executable, "-u", "/kaggle/working/worker.py", str(i), str(n)], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}) for i in range(n)]
codes = [p.wait() for p in procs]; print("종료 코드", codes)


In [ ]:
import glob, csv, os
OUT = "/kaggle/working/results"
rows = []
for p in sorted(glob.glob(f"{OUT}/manifest_*.csv")):
    rows += list(csv.DictReader(open(p, encoding="utf-8")))
fields = ["job_id","body","tone","variant_id","product_id","category","subtype","name","color","file"]
with open(f"{OUT}/manifest.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
manifest_path = f"{OUT}/manifest.csv"; print("생성 완료", len(rows), "장")
from PIL import Image


In [ ]:
# 검수용 미리보기: 카테고리별로 최대 12장
import random
rows = list(csv.DictReader(open(manifest_path, encoding="utf-8")))
for cat in CATEGORIES:
    pick = [r for r in rows if r["category"] == cat][:12]
    if not pick: continue
    sheet = Image.new("RGB", (6*272, ((len(pick)+5)//6)*362), (236, 228, 220))
    for k, r in enumerate(pick):
        im = Image.open(f"{OUT}/{r['file']}").convert("RGBA"); im.thumbnail((272, 362))
        sheet.paste(im, ((k%6)*272, (k//6)*362), im)
    sheet.save(f"{OUT}/preview_{cat}.jpg", quality=85); display(sheet.resize((sheet.width//2, sheet.height//2)))

In [ ]:
with zipfile.ZipFile("/kaggle/working/ondo_fitting_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "/kaggle/working"))
print("완료: Output 탭에서 ondo_fitting_results.zip 을 내려받으세요")